# College Rag Agent


### Loading Necessary Files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.5 MB/s eta 0:00:00


In [ ]:
import pickle
import numpy as np
import faiss

save_dir = "/content/drive/MyDrive/datasets/RAG/model"

with open(f"{save_dir}/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

embeddings = np.load(f"{save_dir}/embeddings.npy")

index = faiss.read_index(
    f"{save_dir}/college_index.faiss"
)

print("Chunks:", len(chunks))
print("Embeddings:", embeddings.shape)
print("FAISS vectors:", index.ntotal)

Chunks: 593
Embeddings: (593, 384)
FAISS vectors: 593


In [ ]:
import os


for file in os.listdir(save_dir):
    path = os.path.join(save_dir, file)
    print(file, os.path.getsize(path), "bytes")

embeddings.npy 910976 bytes
college_index.faiss 910893 bytes
chunks.pkl 444762 bytes


In [ ]:
print("Loaded chunks:", len(chunks))
print(chunks[0])

Loaded chunks: 593
{'text': 'Procedures and policies for maintaining and utilizing physical, academic \nand support facilities - laboratory, library, sports complex, computers, \nclassrooms etc. \n1) Introduction  \nThe college has established system for maintenance and utilisation of computers, classrooms, sports \ngymkhana, laboratories equipment’s and physical facilities. The procedure and policy for the \nmaintenance of various infrastructural facilities are presented in this document. \n2) Purpose of the Policy  \n\uf0b7 The physical and academic facilities are implemented with policies to optimize the use of \nresources based on needs of education, research and administration. \n \uf0b7 The coordination between facility allocation and utilization ensures the optimal usage of \nresources like laboratories, sports gymkhana, library and classrooms inside the campus.  \n\uf0b7 For this Infrastructure and Maintenance Committee of the college plays important role. The \ncommittee revie

### Gemini API Key

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="In one sentence, explain what a university semester is."
)

print(response.text)

A university semester is an academic term, typically lasting 15 to 18 weeks, that divides the school year into two main halves during which students complete a specific set of courses and earn credits.


### Embeddings

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
question = "How many semesters and subjects are there in CSE?"

In [ ]:
query_embedding = embedding_model.encode(
    [question]
).astype("float32")

distances, indices = index.search(
    query_embedding,
    5
)

In [ ]:
for rank, idx in enumerate(indices[0], start=1):
    idx = int(idx)

    print(f"\n--- Result {rank} ---")
    print("File:", chunks[idx]["filename"])
    print("Page:", chunks[idx]["page"])
    print("Text:", chunks[idx]["text"])


--- Result 1 ---
File: 2. Scheme CSE.pdf
Page: 4
Text: 4 | P a g e   
GENERALCOURSESTRUCTURE&THEME 
A. Definition of Credit*: 
 
1Hr. Lecture(L) per week 1Credit 
1Hr.Tutorial(T)per week 1Credit 
1Hr.Practical(P)per week 0.5Credit 
2HoursPractical(P)per week 1Credit 
*Except for mandatory and value added courses 
B. Range of Credits: The total number of credits proposed for the four-year B.Tech. degree in Computer 
Science and Engineering (CSE) is kept as 175. In addition to this, for B.Tech. with Honors & 
specialization/minor degree, the student has to acquire additional 18-20 credits through MOOC 
courses offered at SWAYAM/NPTEL portal. 
C. Structure of UG Program in Computer Science and Engineering (CSE) :The structure of UG 
program in Computer Science and Engineering (CSE)has essentially the following categories of 
courses with the breakup of credits as given: 
 
Sr. 
No. Category Credit Breakup 
for CSE 
1 Humanities and Social Sciences including Management courses       16.5 

In [ ]:
context_parts = []

for idx in indices[0]:
    idx = int(idx)
    chunk = chunks[idx]

    context_parts.append(
        f"Source: {chunk['filename']}, Page: {chunk['page']}\n"
        f"{chunk['text']}"
    )

context = "\n\n---\n\n".join(context_parts)

In [ ]:
question = 'What information is given about hostel registration?'

### Test1

In [ ]:
def ask_rag(question, k=5, show_sources=True):

    # 1. Embed the user's question
    query_embedding = embedding_model.encode(
        [question]
    ).astype("float32")

    # 2. Search the existing FAISS index
    distances, indices = index.search(
        query_embedding,
        k
    )

    # 3. Collect retrieved chunks
    context_parts = []
    sources = []

    for idx in indices[0]:
        idx = int(idx)
        chunk = chunks[idx]

        context_parts.append(
            f"Source: {chunk['filename']}, Page: {chunk['page']}\n"
            f"{chunk['text']}"
        )

        sources.append(
            f"{chunk['filename']} — Page {chunk['page']}"
        )

    # 4. Combine retrieved chunks
    context = "\n\n---\n\n".join(context_parts)

    # 5. Build prompt
    prompt = f"""
You are a college knowledge assistant.

Answer the user's question using ONLY the provided context.


Rules:
- Do not use outside knowledge unless its a general question whose answer remains always same regardless of context.
- You can only visit sites https://www.uietkuk.ac.in and https://kuk.ac.in/index.html
- Do not guess or invent information.
- If the context does not contain enough information, say:
  "I couldn't find this information in the provided documents."
  But if you can find it on given sites accurately, answer the question.
- Give a concise and direct answer.
- Mention the source document and page used. In case you used website then mention that.


CONTEXT:
{context}

QUESTION:
{question}
"""

    # 6. Ask Gemini
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    # 7. Display answer
    print("ANSWER:")
    print(response.text)

    # 8. Display sources
    if show_sources:
        print("\nSOURCES:")
        for source in dict.fromkeys(sources):
            print("-", source)

In [ ]:
ask_rag("What are the laboratory policies?")

ANSWER:
Based on the provided context, the laboratory policies and procedures are as follows:

* **Utilization:** Lab utilization depends on the requirements of various courses, operating under a separate Lab Time-Table where 4 labs are allotted to different programs accordingly.
* **Maintenance & Responsibility:** Respective faculty members, staff, and lab assistants are responsible for maintaining equipment under their purview. Regular maintenance of computers is managed by the Laboratory Administrator, and external expertise is brought in for major repairs.
* **Purchases:** New computer purchases are made via requisition to the Purchase Committee (with quotations sourced from vendors after approval). Components, chemicals, and new equipment for Electronic, Physics, and Chemistry laboratories are purchased yearly through the Purchase Committee.
* **Oversight:** The Infrastructure and Maintenance Committee reviews infrastructure requirements to ensure resources are used optimally for 

In [ ]:
ask_rag("What are the minimum passing marks required?")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 59.259212753s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

## Test2
* To Save tokens, we will firstly retrieve relevant data from source pdfs.
* Then only the relevant info will be passed to model.

In [ ]:
def retrieve(question, k=5, debug=False):
    query_embedding = embedding_model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        idx = int(idx)
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "distance": float(distances[0][rank - 1]),
            "text": chunk["text"],
            "filename": chunk["filename"],
            "page": chunk["page"]
        })

    if debug:
        print(f"Question: {question}\n")
        for r in results:
            print(f"--- Result {r['rank']} ---")
            print(f"Distance: {r['distance']:.4f}")
            print(f"Source: {r['filename']} | Page {r['page']}")
            print(r["text"][:500])
            print()

    return results

In [ ]:
retrieve("What are the laboratory policies?", debug=True)

Question: What are the laboratory policies?

--- Result 1 ---
Distance: 1.0533
Source: LABORATORY Policies-Procedures ,ME.pdf | Page 1
 For this Infrastructure and Maintenance Committee of the college plays important role. The 
committee reviews the requirements of infrastructure  
3) Facilities  
3.1 Laboratory Procedure for Utilisation  
Procedure for Utilisation Maintenance Policy 
The lab utilization is done depending upon the 
requirement of various courses.  
 There is separate Lab Time -Table for all the 
courses. 4 labs are allotted for different programs 
according to their requirements.  
 The respective faculty me

--- Result 2 ---
Distance: 1.1137
Source: LABORATORY Policies-Procedures ,ME.pdf | Page 1
 After approval, the quotations are to be sourced 
from different vendors. 
  Regular maintenance of all computers is done by 
Laboratory Administrator.  For Electronic 
laboratory, Physics Laboratory and Chemistry 
laboratory, the required components ,chemicals 
and ne

[{'rank': 1,
  'distance': 1.053283452987671,
  'text': '\uf0b7 For this Infrastructure and Maintenance Committee of the college plays important role. The \ncommittee reviews the requirements of infrastructure  \n3) Facilities  \n3.1 Laboratory Procedure for Utilisation  \nProcedure for Utilisation Maintenance Policy \nThe lab utilization is done depending upon the \nrequirement of various courses.  \n\uf0b7 There is separate Lab Time -Table for all the \ncourses. 4 labs are allotted for different programs \naccording to their requirements.  \n\uf0b7 The respective faculty members, staff, lab \nassistants are given responsibility to maintain the \nequipment’s under their purview.  \n\uf0b7 All major repairs are identified and external \nexpertise is sought for maintenance of equipment \nwherever necessary. \nPurchase of new computers is done through \nrequisition to Purchase Committee  \n\uf0b7 After approval, the quotations are to be sourced \nfrom different vendors. \n \uf0b7 Regular

In [ ]:
retrieve("How do I register for hostel facilities?", debug=True)

Question: How do I register for hostel facilities?

--- Result 1 ---
Distance: 0.5751
Source: How-to-register-for-Hostel-Facility2.pdf | Page 4
Here you Successfully registered for the Hostel Facility, Now wait for the Admin Approval, once 
your request is approved by Hostel Admin/Warden you can Submit you fees.

--- Result 2 ---
Distance: 0.5837
Source: How-to-register-for-Hostel-Facility2.pdf | Page 1
How to register for Hostel Facility 
 
Step 1 :: Click on URL https://iums.kuk.ac.in/login.htm  & login with your Email/Password 
 
Step 2 :: Click On Facilities>>Hostel>>Hostel Registration. 
 
 
Step 3 :: Select Course Year & Gender.

--- Result 3 ---
Distance: 0.7894
Source: Rules and regulations booklet  2026-27.pdf | Page 14
8.      HOSTEL FACILITIES AND ADMISSION PROCEDURE. 
 
University Authorities make best efforts to provide maximum facilities to the hostellers 
viz. purified cold water, geyser, common room, LED TV with cable broadcasting facility 
channels, Newspapers & magazi

[{'rank': 1,
  'distance': 0.5750519037246704,
  'text': 'Here you Successfully registered for the Hostel Facility, Now wait for the Admin Approval, once \nyour request is approved by Hostel Admin/Warden you can Submit you fees.',
  'filename': 'How-to-register-for-Hostel-Facility2.pdf',
  'page': 4},
 {'rank': 2,
  'distance': 0.5836513042449951,
  'text': 'How to register for Hostel Facility \n \nStep 1 :: Click on URL https://iums.kuk.ac.in/login.htm  & login with your Email/Password \n \nStep 2 :: Click On Facilities>>Hostel>>Hostel Registration. \n \n \nStep 3 :: Select Course Year & Gender.',
  'filename': 'How-to-register-for-Hostel-Facility2.pdf',
  'page': 1},
 {'rank': 3,
  'distance': 0.7894062995910645,
  'text': '8.      HOSTEL FACILITIES AND ADMISSION PROCEDURE. \n \nUniversity Authorities make best efforts to provide maximum facilities to the hostellers \nviz. purified cold water, geyser, common room, LED TV with cable broadcasting facility \nchannels, Newspapers & magaz

In [ ]:
retrieve("Who won football world cup?", debug=True)

Question: Who won football world cup?

--- Result 1 ---
Distance: 1.6745
Source: annual_report_2018.pdf | Page 71
71 | P a g e  
 
 
PANACEA-2018

--- Result 2 ---
Distance: 1.6753
Source: NCC Presentation 5-9-23 (1).pdf | Page 15
ACHIEVEMENT IN 
CAMPS

--- Result 3 ---
Distance: 1.7018
Source: SAE Club Webpage.pdf | Page 5
but
 
also
 
supports
 
India’s
 
goals
 
of
 
agricultural
 
modernization.
 
Students
 
continue
 
to
 
improve
 
the
 
machine,
 
making
 
it
 
more
 
cost-effective
 
and
 
accessible
 
for
 
small
 
and
 
mid-sized
 
farmers.
 
The
 
patent
 
and
 
design
 
grant
 
(one
 
patent
 
and
 
two
 
designs)
 
of
 
the
 
developed
 
machinery
 
marks
 
a
 
signiﬁcant
 
milestone
 
in
 
the
 
club’s
 
history.
 
 
Previous
 
Team
 
Captain:
 
Event
 
Team
 
Name
 
Captain
 
TIFAN
 
2023
 
Team
 
So

--- Result 4 ---
Distance: 1.7147
Source: SAE Club Webpage.pdf | Page 3
(2024-25)
 
 
Our
 
Teams
 
Team
 
Wolf
 
 
Team
 
Wolf
 
is
 
a
 
student-led
 
off-road
 
vehicle


[{'rank': 1,
  'distance': 1.6744941473007202,
  'text': '71 | P a g e  \n \n \nPANACEA-2018',
  'filename': 'annual_report_2018.pdf',
  'page': 71},
 {'rank': 2,
  'distance': 1.675304651260376,
  'text': 'ACHIEVEMENT IN \nCAMPS',
  'filename': 'NCC Presentation 5-9-23 (1).pdf',
  'page': 15},
 {'rank': 3,
  'distance': 1.701764464378357,
  'text': 'but\n \nalso\n \nsupports\n \nIndia’s\n \ngoals\n \nof\n \nagricultural\n \nmodernization.\n \nStudents\n \ncontinue\n \nto\n \nimprove\n \nthe\n \nmachine,\n \nmaking\n \nit\n \nmore\n \ncost-effective\n \nand\n \naccessible\n \nfor\n \nsmall\n \nand\n \nmid-sized\n \nfarmers.\n \nThe\n \npatent\n \nand\n \ndesign\n \ngrant\n \n(one\n \npatent\n \nand\n \ntwo\n \ndesigns)\n \nof\n \nthe\n \ndeveloped\n \nmachinery\n \nmarks\n \na\n \nsigniﬁcant\n \nmilestone\n \nin\n \nthe\n \nclub’s\n \nhistory.\n \n \nPrevious\n \nTeam\n \nCaptain:\n \nEvent\n \nTeam\n \nName\n \nCaptain\n \nTIFAN\n \n2023\n \nTeam\n \nSowertact\n \nPankaj\n \nJhangra\n

### Checking all available models

In [ ]:
available_models = []

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        available_models.append(model.name)

for model in available_models:
    print(model)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [ ]:
def ask_rag(question, k=5, debug=False):
    # ---------- RETRIEVAL ----------
    results = retrieve(question, k=k, debug=debug)

    # Build context
    context_parts = []

    for r in results:
        context_parts.append(
            f"""SOURCE:
Document: {r['filename']}
Page: {r['page']}

CONTENT:
{r['text']}"""
        )

    context = "\n\n---\n\n".join(context_parts)

    # ---------- PROMPT ----------
    prompt = f"""
You are a College Knowledge Assistant.

Answer the user's question using ONLY the information provided
in the CONTEXT below.

Rules:
- Do not use outside knowledge unless its a general question whose answer remains always same regardless of context.
- You can only visit sites https://www.uietkuk.ac.in and https://kuk.ac.in/index.html
- Do not guess or invent information.
- If the context does not contain enough information, say:
  "I couldn't find this information in the provided documents."
  But if you can find it on given sites accurately, answer the question.
- Give a concise and direct answer.
- Mention the source document and page used. In case you used website then mention that.

- When the answer is supported, mention the relevant document
   and page number.
- If multiple sources are relevant, use them together.

CONTEXT:
{context}

USER QUESTION:
{question}
"""

    # ---------- MODEL 1 ----------
    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        answer = response.text
        model_used = "Gemini 3.6 Flash"

    # ---------- FALLBACK ----------
    except Exception as e:
        print("Primary model failed. Trying fallback...")

        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )

            answer = response.text
            model_used = "Gemini 3.5 Flash-Lite"

        except Exception as fallback_error:
            return {
                "answer": "Sorry, the AI service is currently unavailable.",
                "sources": [],
                "model": None,
                "error": str(fallback_error)
            }

    # ---------- SOURCES ----------
    sources = []

    for r in results:
        source = f"{r['filename']} — Page {r['page']}"

        if source not in sources:
            sources.append(source)

    # ---------- RESULT ----------
    result = {
        "answer": answer,
        "sources": sources,
        "model": model_used
    }

    # ---------- DISPLAY ----------
    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    for source in sources:
        print("-", source)

    print(f"\nMODEL: {model_used}")

    return result

In [ ]:
ask_rag("How do I register for hostel facilities?")

Primary model failed. Trying fallback...

ANSWER:
To register for hostel facilities, follow these steps:

1. Go to the University web portal at https://iums.kuk.ac.in and log in with your Email/Password.
2. Click on **Facilities** >> **Hostel** >> **Hostel Registration**.
3. Select your Course Year and Gender.
4. Add Visitor Details.
5. Fill in all other required details.
6. Click on **Self Declaration** and then click on **Register**.

**Source:** 
- Document: `How-to-register-for-Hostel-Facility2.pdf` (Pages 1 and 3)
- Document: `Rules and regulations booklet  2026-27.pdf` (Page 14)

SOURCES:
- How-to-register-for-Hostel-Facility2.pdf — Page 4
- How-to-register-for-Hostel-Facility2.pdf — Page 1
- Rules and regulations booklet  2026-27.pdf — Page 14
- How-to-register-for-Hostel-Facility2.pdf — Page 3
- Rules and regulations booklet  2026-27.pdf — Page 21

MODEL: Gemini 3.5 Flash-Lite


{'answer': 'To register for hostel facilities, follow these steps:\n\n1. Go to the University web portal at https://iums.kuk.ac.in and log in with your Email/Password.\n2. Click on **Facilities** >> **Hostel** >> **Hostel Registration**.\n3. Select your Course Year and Gender.\n4. Add Visitor Details.\n5. Fill in all other required details.\n6. Click on **Self Declaration** and then click on **Register**.\n\n**Source:** \n- Document: `How-to-register-for-Hostel-Facility2.pdf` (Pages 1 and 3)\n- Document: `Rules and regulations booklet  2026-27.pdf` (Page 14)',
 'sources': ['How-to-register-for-Hostel-Facility2.pdf — Page 4',
  'How-to-register-for-Hostel-Facility2.pdf — Page 1',
  'Rules and regulations booklet  2026-27.pdf — Page 14',
  'How-to-register-for-Hostel-Facility2.pdf — Page 3',
  'Rules and regulations booklet  2026-27.pdf — Page 21'],
 'model': 'Gemini 3.5 Flash-Lite'}

# Final Testing

### ask_rag() function
* It firstly retrieves relevant data and builds relevant context.
* Now we have the prompt.
* we have primary model Gemini-3.6-flash but as it is often in high demand we have a fallback model gemini-3.5-flash-lite.
* Now it will answer the query and will also show name and page no. of information of relevant source pdf.

In [ ]:
PRIMARY_MODEL = "gemini-3.6-flash"
FALLBACK_MODEL = "gemini-3.5-flash-lite"

In [ ]:
def ask_rag(question, k=5, debug=False):

    # ---------- RETRIEVE ----------
    results = retrieve(question, k=k, debug=debug)

    # ---------- BUILD CONTEXT ----------
    context_parts = []

    for r in results:
        context_parts.append(
            f"""SOURCE
Document: {r['filename']}
Page: {r['page']}

CONTENT
{r['text']}"""
        )

    context = "\n\n---\n\n".join(context_parts)

    # ---------- PROMPT ----------
    prompt = f"""
You are a College Knowledge Assistant.

Answer the user's question using ONLY the information
contained in the provided sources.

Rules:
- If the question is unrelated to the college or the provided documents,
   DO NOT answer it using your general knowledge, say:
   "I'm here to answer questions about the college and its documents.
   Please ask a relevant question about the college, academics, courses,
   facilities, departments, rules, procedures, or other information
   covered in the provided documents."
- Do not guess or invent information.
- Do not use outside knowledge unless its a general question related to college and student whose answer remains always same regardless of context.
- You can only visit sites https://www.uietkuk.ac.in and https://kuk.ac.in/index.html
- Do not guess or invent information.
- If the context does not contain enough information, say:
  "I couldn't find this information in the provided documents."
  But if you can find it on given sites accurately, answer the question.
- Give a concise and direct answer.
- Mention the source document and page used. In case you used website then mention that.

- When the answer is supported, mention the relevant document
   and page number.
- If multiple sources are relevant, use them together.

PROVIDED SOURCES:

{context}

USER QUESTION:

{question}
"""

    # ---------- PRIMARY MODEL ----------
    try:
        response = client.models.generate_content(
            model=PRIMARY_MODEL,
            contents=prompt
        )

        answer = response.text
        model_used = PRIMARY_MODEL

    # ---------- FALLBACK MODEL ----------
    except Exception as primary_error:

        print("Primary model failed.")
        print("Using fallback model...")

        try:
            response = client.models.generate_content(
                model=FALLBACK_MODEL,
                contents=prompt
            )

            answer = response.text
            model_used = FALLBACK_MODEL

        except Exception as fallback_error:

            print("Both models failed.")

            return {
                "answer": "Sorry, the AI service is currently unavailable.",
                "sources": [],
                "model": None
            }

    # ---------- SOURCES ----------
    sources = []

    for r in results:
        source = f"{r['filename']} — Page {r['page']}"

        if source not in sources:
            sources.append(source)

    # ---------- DISPLAY ----------
    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    for source in sources:
        print("-", source)

    print("\nMODEL USED:")
    print(model_used)

    return {
        "answer": answer,
        "sources": sources,
        "model": model_used
    }

In [ ]:
ask_rag("How many semesters are there in CSE?")


ANSWER:
Based on the provided documents, there are **8 semesters** in the four-year B.Tech. degree program in Computer Science and Engineering (CSE).

*Source:*
- **2. Scheme CSE.pdf** (Page 2, Page 4)

SOURCES:
- 2. Scheme CSE.pdf — Page 4
- SchemeCSE_AIML.pdf — Page 2
- 2. Scheme CSE.pdf — Page 2
- 2. Scheme CSE.pdf — Page 9
- SchemeCSE_AIML.pdf — Page 4

MODEL USED:
gemini-3.6-flash


{'answer': 'Based on the provided documents, there are **8 semesters** in the four-year B.Tech. degree program in Computer Science and Engineering (CSE).\n\n*Source:*\n- **2. Scheme CSE.pdf** (Page 2, Page 4)',
 'sources': ['2. Scheme CSE.pdf — Page 4',
  'SchemeCSE_AIML.pdf — Page 2',
  '2. Scheme CSE.pdf — Page 2',
  '2. Scheme CSE.pdf — Page 9',
  'SchemeCSE_AIML.pdf — Page 4'],
 'model': 'gemini-3.6-flash'}

In [ ]:
test_questions = [
    # CSE / academics
    "How many semesters are there in B.Tech CSE?",
    "What is the credit structure of the CSE programme?",
    "What are the different categories of courses in CSE?",
    "What information is given about the CSE curriculum?",

    # College facilities
    "What facilities are mentioned in the college infrastructure?",
    "What information is provided about the college library?",
    "What are the laboratory policies and procedures?",

    # Hostel
    "How do students register for hostel facilities?",
    "What information is provided about hostel facilities?",

    # Administration
    "What is the procedure for verification of educational qualifications?",
    "What information is provided by the examination cell?",
    "What does the mandatory disclosure document contain?",

    # Student activities
    "What information is provided about NCC?",
    "What is mentioned about the SAE Club?",
    "What information is provided about college festivals?",

    # Academic schedule
    "What is the academic schedule for 2025-26?",
    "What information is given about the 2025-26 session?",

    # Out-of-document questions
    "What is the hostel mess fee?",
    "What is tomorrow's weather in Kurukshetra?",
    "Who is the current Prime Minister of India?"
]

In [ ]:
for question in test_questions:
    results = retrieve(question, k=5)

    print("\n" + "="*80)
    print("QUESTION:", question)
    print("TOP SOURCE:", results[0]["filename"])
    print("PAGE:", results[0]["page"])


QUESTION: How many semesters are there in B.Tech CSE?
TOP SOURCE: 2. Scheme CSE.pdf
PAGE: 4

QUESTION: What is the credit structure of the CSE programme?
TOP SOURCE: 2. Scheme CSE.pdf
PAGE: 4

QUESTION: What are the different categories of courses in CSE?
TOP SOURCE: 2. Scheme CSE.pdf
PAGE: 4

QUESTION: What information is given about the CSE curriculum?
TOP SOURCE: 2. Scheme CSE.pdf
PAGE: 4

QUESTION: What facilities are mentioned in the college infrastructure?
TOP SOURCE: LABORATORY Policies-Procedures ,ME.pdf
PAGE: 1

QUESTION: What information is provided about the college library?
TOP SOURCE: annual_report_2018.pdf
PAGE: 11

QUESTION: What are the laboratory policies and procedures?
TOP SOURCE: LABORATORY Policies-Procedures ,ME.pdf
PAGE: 1

QUESTION: How do students register for hostel facilities?
TOP SOURCE: How-to-register-for-Hostel-Facility2.pdf
PAGE: 1

QUESTION: What information is provided about hostel facilities?
TOP SOURCE: Rules and regulations booklet  2026-27.pdf
PAG

### Checking retrieval relevance threshold.

In [ ]:
questions = [
    "How many semesters are there in CSE?",
    "What are the laboratory policies?",
    "How do students register for hostel facilities?",
    "Who is the current Prime Minister of India?",
    "What is tomorrow's weather in Kurukshetra?",
    "What is the hostel mess fee?"
]

for q in questions:
    results = retrieve(q, k=5)

    print("\n" + "=" * 70)
    print("QUESTION:", q)

    for r in results[:3]:
        print(
            f"Distance: {r['distance']:.4f} | "
            f"{r['filename']} | Page {r['page']}"
        )


QUESTION: How many semesters are there in CSE?
Distance: 0.6056 | 2. Scheme CSE.pdf | Page 4
Distance: 0.6905 | SchemeCSE_AIML.pdf | Page 2
Distance: 0.6905 | 2. Scheme CSE.pdf | Page 2

QUESTION: What are the laboratory policies?
Distance: 1.0533 | LABORATORY Policies-Procedures ,ME.pdf | Page 1
Distance: 1.1137 | LABORATORY Policies-Procedures ,ME.pdf | Page 1
Distance: 1.1349 | mandatory Disclosure.pdf | Page 5

QUESTION: How do students register for hostel facilities?
Distance: 0.6433 | How-to-register-for-Hostel-Facility2.pdf | Page 1
Distance: 0.6913 | Rules and regulations booklet  2026-27.pdf | Page 14
Distance: 0.7684 | Rules and regulations booklet  2026-27.pdf | Page 15

QUESTION: Who is the current Prime Minister of India?
Distance: 0.9983 | Revised Executive Committee 2025-26.pdf | Page 3
Distance: 1.1288 | Revised Executive Committee 2025-26.pdf | Page 6
Distance: 1.1399 | Revised Executive Committee 2025-26.pdf | Page 1

QUESTION: What is tomorrow's weather in Kurukshet

In [ ]:
ask_rag('Whos the prime minister of India?')


ANSWER:
I'm here to answer questions about the college and its documents. Please ask a relevant question about the college, academics, courses, facilities, departments, rules, procedures, or other information covered in the provided documents.

SOURCES:
- Revised Executive Committee 2025-26.pdf — Page 3
- Revised Executive Committee 2025-26.pdf — Page 1
- Revised Executive Committee 2025-26.pdf — Page 6
- Revised Executive Committee 2025-26.pdf — Page 2
- annual_report_2018.pdf — Page 31

MODEL USED:
gemini-3.6-flash


{'answer': "I'm here to answer questions about the college and its documents. Please ask a relevant question about the college, academics, courses, facilities, departments, rules, procedures, or other information covered in the provided documents.",
 'sources': ['Revised Executive Committee 2025-26.pdf — Page 3',
  'Revised Executive Committee 2025-26.pdf — Page 1',
  'Revised Executive Committee 2025-26.pdf — Page 6',
  'Revised Executive Committee 2025-26.pdf — Page 2',
  'annual_report_2018.pdf — Page 31'],
 'model': 'gemini-3.6-flash'}

In [ ]:
ask_rag('What is the location of IIT Delhi?')


ANSWER:
I couldn't find this information in the provided documents.

SOURCES:
- mandatory Disclosure.pdf — Page 4
- annual_report_2018.pdf — Page 98
- annual_report_2018.pdf — Page 64
- annual_report_2018.pdf — Page 101
- annual_report_2018.pdf — Page 18

MODEL USED:
gemini-3.6-flash


{'answer': "I couldn't find this information in the provided documents.",
 'sources': ['mandatory Disclosure.pdf — Page 4',
  'annual_report_2018.pdf — Page 98',
  'annual_report_2018.pdf — Page 64',
  'annual_report_2018.pdf — Page 101',
  'annual_report_2018.pdf — Page 18'],
 'model': 'gemini-3.6-flash'}

In [ ]:
ask_rag('Who is the vice chancellor of University?')


ANSWER:
According to the provided documents, the Vice-Chancellor of Kurukshetra University is Dr. Kailash Chandra Sharma.

**Source:**
* Document: `annual_report_2018.pdf`, Page 7

SOURCES:
- annual_report_2018.pdf — Page 7
- annual_report_2018.pdf — Page 67
- Schedule_session_2025-26.pdf — Page 2

MODEL USED:
gemini-3.6-flash


{'answer': 'According to the provided documents, the Vice-Chancellor of Kurukshetra University is Dr. Kailash Chandra Sharma.\n\n**Source:**\n* Document: `annual_report_2018.pdf`, Page 7',
 'sources': ['annual_report_2018.pdf — Page 7',
  'annual_report_2018.pdf — Page 67',
  'Schedule_session_2025-26.pdf — Page 2'],
 'model': 'gemini-3.6-flash'}

In [ ]:
ask_rag('What are the placement opprtunities?')


ANSWER:
Based on the provided documents, the Training and Placement Cell provides placement opportunities by:

* Conducting placement drives and inviting reputed companies for campus recruitment for undergraduate and postgraduate students.
* Imparting on-the-job training opportunities to help students learn new skills and secure job positions in reputed corporate houses.
* Gathering information about job fairs and recruitment advertisements, and providing career guidance. 
* Mentioned participating companies include Deftsoft Informatics.

**Source Document:** `annual_report_2018.pdf` (Pages 62, 64)

SOURCES:
- annual_report_2018.pdf — Page 61
- annual_report_2018.pdf — Page 2
- annual_report_2018.pdf — Page 64
- annual_report_2018.pdf — Page 5
- annual_report_2018.pdf — Page 62

MODEL USED:
gemini-3.6-flash


{'answer': 'Based on the provided documents, the Training and Placement Cell provides placement opportunities by:\n\n* Conducting placement drives and inviting reputed companies for campus recruitment for undergraduate and postgraduate students.\n* Imparting on-the-job training opportunities to help students learn new skills and secure job positions in reputed corporate houses.\n* Gathering information about job fairs and recruitment advertisements, and providing career guidance. \n* Mentioned participating companies include Deftsoft Informatics.\n\n**Source Document:** `annual_report_2018.pdf` (Pages 62, 64)',
 'sources': ['annual_report_2018.pdf — Page 61',
  'annual_report_2018.pdf — Page 2',
  'annual_report_2018.pdf — Page 64',
  'annual_report_2018.pdf — Page 5',
  'annual_report_2018.pdf — Page 62'],
 'model': 'gemini-3.6-flash'}

In [ ]:
ask_rag('What are the medical facilities?')


ANSWER:
Based on the provided documents, the available medical facilities include:

* **University Health Centre Services:** Out-door and indoor patient treatment, allopathic medicines, and pre-joining medical examinations for students and staff.
* **Diagnostic & Laboratory Facilities:** Computerized ECG, EEG, Spirometry, X-Ray, and Laboratory facilities (including Computerized Bio-Chemistry Auto Analyzer).
* **Specialized Units & Emergency:** Physiotherapy unit and ambulance facility to shift patients to another hospital.
* **Hostel Medical Support:** Wardens assist residents during medical exigencies, and referrals to private hospitals for girl residents are managed through the Resident Medical Officer of the University Health Centre.

**Sources:**
* `annual_report_2018.pdf`, Page 13
* `Rules and regulations booklet  2026-27.pdf`, Page 17

SOURCES:
- annual_report_2018.pdf — Page 13
- Rules and regulations booklet  2026-27.pdf — Page 17
- LABORATORY Policies-Procedures ,ME.pdf — Pag

{'answer': 'Based on the provided documents, the available medical facilities include:\n\n* **University Health Centre Services:** Out-door and indoor patient treatment, allopathic medicines, and pre-joining medical examinations for students and staff.\n* **Diagnostic & Laboratory Facilities:** Computerized ECG, EEG, Spirometry, X-Ray, and Laboratory facilities (including Computerized Bio-Chemistry Auto Analyzer).\n* **Specialized Units & Emergency:** Physiotherapy unit and ambulance facility to shift patients to another hospital.\n* **Hostel Medical Support:** Wardens assist residents during medical exigencies, and referrals to private hospitals for girl residents are managed through the Resident Medical Officer of the University Health Centre.\n\n**Sources:**\n* `annual_report_2018.pdf`, Page 13\n* `Rules and regulations booklet  2026-27.pdf`, Page 17',
 'sources': ['annual_report_2018.pdf — Page 13',
  'Rules and regulations booklet  2026-27.pdf — Page 17',
  'LABORATORY Policies-Pr